# Phase 2: Data Preprocessing & Feature Engineering
## Credit Risk & Loan Decision Intelligence Platform

---

### Notebook Overview & Progression:
This notebook covers the complete end-to-end **Phase 2** pipeline preparing raw credit data for machine learning modeling:
- **Part 1 (Step 1)**: ML Dataset Definition + Train/Test Split
- **Part 2 (Step 2)**: Feature Preprocessing (StandardScaler, OneHotEncoder, ColumnTransformer)
- **Part 3 (Step 3)**: Feature Engineering & Model-Input Validation (Integrity Checks, Redundancy Audit & Model-Ready Gates)

---

> **Important Notice — Synthetic Data (50 records):**  
> The dataset used in this notebook (`credit_risk_50.csv`) contains **50 synthetic records** created specifically for learning and developing the end-to-end Data Science workflow.  
> The 80/20 train/test split yields 40 training records and 10 testing records. While 10 test records are small for statistical inference, this demonstrates the exact enterprise workflow used for large-scale credit underwriting models.

---

### Core Architectural Principle: Identity Risk ≠ Credit Default Risk
- **Verification / Product Signals** (`pan_verified`, `kyc_verified`, `mobile_verified`, `identity_match_status`):
  - Handled in `src/verification/` as **Eligibility Gates**. If identity/KYC fails, the application is rejected or routed for manual review **before** ML scoring.
- **Credit Default Risk Signals** (Credit score, DTI, payment history, banking cashflows):
  - Fed into the ML pipeline to estimate **Probability of Default (PD)**.

---
# ============================================================
# PART 1: PHASE 2 — STEP 1: ML Dataset Definition & Split
# ============================================================


### 1.1 Environment Setup & Imports

#### WHAT are we doing?
We import core data processing libraries (`pandas`, `numpy`) and `scikit-learn` model selection and preprocessing components.

#### WHY are we doing it?
To establish a reproducible, standardized scientific computing environment.

#### WHAT does the output mean?
Verifies that library versions are loaded and ready.

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Display settings for clean table outputs
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Pandas version: {pd.__version__}")


Pandas version: 3.0.3


### 1.2 Load Dataset & Inspect Structure

#### WHAT are we doing?
We load `credit_risk_50.csv` from `data/raw/` and inspect its shape and column data types.

#### WHY are we doing it?
To verify data ingestion and list raw column definitions.

#### WHAT does the output mean?
The raw dataset contains **50 rows** and **16 columns**.

In [4]:
# Load raw dataset
data_path = '../data/raw/credit_risk_50.csv'
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Columns list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:25s} ({df[col].dtype})")


Dataset Shape: 50 rows, 16 columns

Columns list:
   1. applicant_id              (int64)
   2. age                       (int64)
   3. employment_type           (str)
   4. monthly_income            (int64)
   5. loan_amount               (int64)
   6. loan_term_months          (int64)
   7. existing_emi              (int64)
   8. credit_score              (int64)
   9. credit_utilization_pct    (int64)
  10. late_payment_count        (int64)
  11. previous_defaults         (int64)
  12. employment_years          (int64)
  13. average_bank_balance      (int64)
  14. monthly_expenses          (int64)
  15. debt_to_income_ratio      (float64)
  16. default                   (int64)


### 1.3 Feature Taxonomy & Target Definition

| Category | Columns | Role in System |
|---|---|---|
| **Identifier** | `applicant_id` | Database surrogate key. **Must be excluded** from ML. |
| **Target ($y$)** | `default` | Binary label (0 = Repaid, 1 = Defaulted). |
| **Categorical ML Feature** | `employment_type` | Job classification (`Salaried`, `Self-employed`, `Business`). |
| **Numerical ML Features (13 cols)** | `age`, `monthly_income`, `loan_amount`, `loan_term_months`, `existing_emi`, `credit_score`, `credit_utilization_pct`, `late_payment_count`, `previous_defaults`, `employment_years`, `average_bank_balance`, `monthly_expenses`, `debt_to_income_ratio` | Financial capacity and bureau metrics. |
| **Verification / Product Signals** | `pan_verified`, `kyc_verified`, `mobile_verified`, etc. | Outside ML vector; used in eligibility workflow. |

---

#### WHAT are we doing?
We extract target $y$ (`default`) and construct feature matrix $X$ by dropping `applicant_id` and `default`.

#### WHY are we doing it?
- `applicant_id` is an arbitrary database key that provides zero generalizable predictive signal and risks spurious correlation / memorization.
- `default` is our ground truth target.

#### WHAT does the output mean?
$X$ contains 14 features across 50 records; $y$ contains 50 binary labels (31 non-default, 19 default).

In [5]:
# Target definition
target_col = 'default'
id_col = 'applicant_id'

y = df[target_col].copy()
ml_feature_cols = [col for col in df.columns if col not in [id_col, target_col]]
X = df[ml_feature_cols].copy()

# Assertions
assert len(X) == len(y), "Row count mismatch between X and y"
assert id_col not in X.columns, f"'{id_col}' must not be in X"
assert target_col not in X.columns, f"'{target_col}' must not be in X"

print(f"X shape: {X.shape} (50 rows, 14 features)")
print(f"y shape: {y.shape} (50 labels)\n")

counts = y.value_counts().rename({0: '0 (Non-default)', 1: '1 (Default)'})
pcts = (y.value_counts(normalize=True) * 100).rename({0: '0 (Non-default)', 1: '1 (Default)'})
print("Overall Target Distribution:")
print(pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)}))


X shape: (50, 14) (50 rows, 14 features)
y shape: (50,) (50 labels)

Overall Target Distribution:
                 Count  Percentage (%)
default                               
0 (Non-default)     31            62.0
1 (Default)         19            38.0


### 1.4 Stratified Train / Test Split

#### WHAT are we doing?
We split $X$ and $y$ into training (80% = 40 rows) and testing (20% = 10 rows) partitions using `stratify=y` and `random_state=42`.

#### WHY are we doing it?
- **Stratification**: Preserves the ~62.5% non-default / 37.5% default ratio in both train and test splits, preventing sampling bias.
- **Leakage Prevention**: Splitting before fitting any preprocessing guarantees the test set remains strictly unseen.

#### WHAT does the output mean?
The dataset is cleanly divided into `X_train` (40 rows) and `X_test` (10 rows) with preserved class distributions.

In [6]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape : {X_test.shape}  | y_test shape : {y_test.shape}\n")

summary_table = pd.DataFrame({
    'Dataset': ['Full Dataset', 'Train Set (80%)', 'Test Set (20%)'],
    'Total Rows': [len(y), len(y_train), len(y_test)],
    'Non-Default (0)': [(y==0).sum(), (y_train==0).sum(), (y_test==0).sum()],
    'Default (1)': [(y==1).sum(), (y_train==1).sum(), (y_test==1).sum()],
    'Non-Default %': [round((y==0).mean()*100, 2), round((y_train==0).mean()*100, 2), round((y_test==0).mean()*100, 2)],
    'Default %': [round((y==1).mean()*100, 2), round((y_train==1).mean()*100, 2), round((y_test==1).mean()*100, 2)]
})
print("=== Target Distribution Verification ===")
display(summary_table)


X_train shape: (40, 14) | y_train shape: (40,)
X_test shape : (10, 14)  | y_test shape : (10,)

=== Target Distribution Verification ===


,Dataset,Total Rows,Non-Default (0),Default (1),Non-Default %,Default %
0,Full Dataset,50,31,19,62.0,38.0
1,Train Set (80%),40,25,15,62.5,37.5
2,Test Set (20%),10,6,4,60.0,40.0


---
# ============================================================
# PART 2: PHASE 2 — STEP 2: Feature Preprocessing
# ============================================================


### 2.1 Preprocessing Pipeline Architecture

#### 1. Why Preprocessing is Needed:
- **Categorical Strings**: `employment_type` contains text labels (`"Salaried"`, `"Business"`, `"Self-employed"`). Mathematical models cannot calculate dot products ($w^T x$) with strings.
- **Disparate Numeric Scales**: `loan_amount` (₹1,20,000 to ₹7,00,000) vs. `debt_to_income_ratio` (0.18 to 0.55). Unscaled features cause erratic optimization updates and distort regularized models ($L_1/L_2$).

#### 2. Components:
- **`StandardScaler`**: Scales 13 numerical columns: $z = \frac{x - \mu}{\sigma}$.
- **`OneHotEncoder`**: Converts `employment_type` to 3 binary columns with `handle_unknown='ignore'`.
- **`ColumnTransformer`**: Combines both transformations, fitted **strictly on `X_train`**.

In [7]:
# Define feature groupings
numerical_cols = [
    'age', 'monthly_income', 'loan_amount', 'loan_term_months', 'existing_emi',
    'credit_score', 'credit_utilization_pct', 'late_payment_count', 'previous_defaults',
    'employment_years', 'average_bank_balance', 'monthly_expenses', 'debt_to_income_ratio'
]
categorical_cols = ['employment_type']

# Construct ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    verbose_feature_names_out=False
)

# Fit preprocessor strictly on X_train
preprocessor.fit(X_train)
transformed_feature_names = list(preprocessor.get_feature_names_out())

# Transform X_train and X_test using fitted parameters
X_train_processed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=transformed_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=transformed_feature_names,
    index=X_test.index
)

print(f"X_train shape: Raw {X_train.shape} --> Processed {X_train_processed.shape}")
print(f"X_test shape : Raw {X_test.shape}  --> Processed {X_test_processed.shape}")
print(f"Generated feature count: {len(transformed_feature_names)}")


X_train shape: Raw (40, 14) --> Processed (40, 16)
X_test shape : Raw (10, 14)  --> Processed (10, 16)
Generated feature count: 16


### 2.2 Scaling & Encoding Verification

#### WHAT are we doing?
We verify that `X_train_processed` numerical features have $\mu \approx 0.0$ and $\sigma = 1.0$, while `X_test_processed` features maintain their natural variation.

#### WHY are we doing it?
To prove that transformations were fitted exclusively on training data without test set leakage.

#### WHAT does the output mean?
Scaling and one-hot encoding execute flawlessly across both partitions.

In [8]:
# Check scaling statistics
scaling_check = pd.DataFrame({
    'Train Mean (~0)': X_train_processed[numerical_cols].mean().round(4),
    'Train Std (1.0)': X_train_processed[numerical_cols].std(ddof=0).round(4),
    'Test Mean (Natural)': X_test_processed[numerical_cols].mean().round(4),
    'Test Std (Natural)': X_test_processed[numerical_cols].std(ddof=0).round(4)
})
print("=== Scaling Check Summary ===")
display(scaling_check.head(6))


=== Scaling Check Summary ===


,Train Mean (~0),Train Std (1.0),Test Mean (Natural),Test Std (Natural)
age,0.0,1.0,-0.1378,0.9177
monthly_income,-0.0,1.0,-0.0609,0.9837
loan_amount,0.0,1.0,-0.0269,0.9815
loan_term_months,0.0,1.0,0.0234,0.9533
existing_emi,0.0,1.0,-0.0448,0.8955
credit_score,0.0,1.0,-0.0518,1.0604


---
# ============================================================
# PART 3: PHASE 2 — STEP 3: Feature Engineering & Model-Input Validation
# ============================================================


## 1. Why Model-Input Validation is Needed

In enterprise machine learning systems, **garbage in leads to catastrophic failure out**.

Before feeding processed matrices (`X_train_processed`, `X_test_processed`) to ML algorithms, we must perform an exhaustive, rigorous validation audit. A silent flaw at this stage—such as a single `NaN`, an accidental target leakage, a misaligned column order, or an unhandled infinite value—will either crash model training or produce deceptively high scores that fail completely in production.

---

### Key Validation Checks:
1. **Train/Test Feature Consistency**: Identical columns, ordering, and names.
2. **Missing-Value Gate**: Zero `NaN` or `None` values.
3. **Infinite-Value Gate**: Zero `+inf` or `-inf` values.
4. **Data Type Consistency**: 100% numerical (`float64`), no leftover strings/objects.
5. **Target Separation**: Target `default` is strictly quarantined in `y_train` / `y_test`.
6. **Identifier Exclusion**: `applicant_id` is 100% excluded.
7. **Identity Risk Separation**: KYC/verification flags remain in the eligibility layer.
8. **Feature Redundancy Audit**: Analysis of correlated indicators without premature deletion.
9. **Final Automated Assertions**: Hard gates that fail loudly if the dataset is compromised.

## 2. Train / Test Feature Consistency

### WHAT are we checking?
We verify that `X_train_processed` and `X_test_processed`:
- Have the **exact same number of columns** (16 features).
- Have the **exact same column names**.
- Have the **exact same feature ordering**.

### WHY are we checking it?
Machine learning models (like scikit-learn estimators or XGBoost) treat features strictly as ordered column indices in a matrix. If columns are scrambled between train and test, the model will apply weights learned for one feature (e.g., `credit_score`) to a completely different feature (e.g., `monthly_income`), destroying evaluation validity.

### WHAT does the result mean?
Both matrices have identical, synchronized feature schemas.

In [9]:
# Check 1: Feature Schema Consistency
train_cols = list(X_train_processed.columns)
test_cols = list(X_test_processed.columns)

cols_match = (train_cols == test_cols)
col_count_match = (len(train_cols) == len(test_cols) == 16)

print(f"Train columns count       : {len(train_cols)}")
print(f"Test columns count        : {len(test_cols)}")
print(f"Identical column ordering : {cols_match}")
print(f"Expected 16 columns       : {col_count_match}\n")

assert cols_match, "CRITICAL ERROR: Train and Test column schemas do not match!"
assert col_count_match, f"CRITICAL ERROR: Expected 16 features, got {len(train_cols)}"

print("Train/Test Feature Consistency Check: PASSED")


Train columns count       : 16
Test columns count        : 16
Identical column ordering : True
Expected 16 columns       : True

Train/Test Feature Consistency Check: PASSED


## 3. Missing-Value (`NaN`) Check

### WHAT are we checking?
We inspect both `X_train_processed` and `X_test_processed` for any `NaN`, `None`, or missing values.

### WHY are we checking it?
Standard machine learning algorithms (such as Logistic Regression, Support Vector Machines, and standard Neural Networks) cannot compute mathematical dot products with `NaN`. Passing `NaN` values causes either immediate runtime errors or silent propagation of invalid gradients.

### WHAT does the result mean?
Both training and test matrices contain **0 missing values**.

In [10]:
# Check 2: Missing Values (NaN)
train_nans = X_train_processed.isna().sum().sum()
test_nans = X_test_processed.isna().sum().sum()

print(f"Total NaN values in X_train_processed: {train_nans}")
print(f"Total NaN values in X_test_processed : {test_nans}\n")

assert train_nans == 0, f"CRITICAL ERROR: Found {train_nans} NaNs in X_train_processed!"
assert test_nans == 0, f"CRITICAL ERROR: Found {test_nans} NaNs in X_test_processed!"

print("Missing-Value Check: PASSED (0 NaNs found)")


Total NaN values in X_train_processed: 0
Total NaN values in X_test_processed : 0

Missing-Value Check: PASSED (0 NaNs found)


## 4. Infinite-Value Check (`+inf` / `-inf`)

### WHAT are we checking?
We verify that no mathematical operation (such as division by zero in ratios or exponent overflow) has introduced positive or negative infinity (`+inf`, `-inf`) into either processed matrix.

### WHY are we checking it?
Infinite values cause gradient divergence, weight explosion to `NaN`, and complete failure in linear models, trees, and neural networks.

### WHAT does the result mean?
Both processed matrices contain **0 infinite values**.

In [11]:
# Check 3: Infinite Values (+inf / -inf)
train_infs = np.isinf(X_train_processed.values).sum()
test_infs = np.isinf(X_test_processed.values).sum()

print(f"Total Infinite values in X_train_processed: {train_infs}")
print(f"Total Infinite values in X_test_processed : {test_infs}\n")

assert train_infs == 0, f"CRITICAL ERROR: Found {train_infs} infinite values in X_train_processed!"
assert test_infs == 0, f"CRITICAL ERROR: Found {test_infs} infinite values in X_test_processed!"

print("Infinite-Value Check: PASSED (0 infinite values found)")


Total Infinite values in X_train_processed: 0
Total Infinite values in X_test_processed : 0

Infinite-Value Check: PASSED (0 infinite values found)


## 5. Numerical Data Type Check

### WHAT are we checking?
We inspect the `dtype` of every column in `X_train_processed` and `X_test_processed` to confirm all features are floating-point numbers (`float64`).

### WHY are we checking it?
To confirm that no raw string categories (`"Salaried"`), object dtypes, or non-numeric types remain after encoding.

### WHAT does the result mean?
All 16 features in both partitions are clean, continuous floating-point numbers (`float64`).

In [12]:
# Check 4: Data Types
train_dtypes = X_train_processed.dtypes.value_counts().to_dict()
test_dtypes = X_test_processed.dtypes.value_counts().to_dict()

print(f"X_train_processed data types: {train_dtypes}")
print(f"X_test_processed data types : {test_dtypes}\n")

all_train_numeric = all(np.issubdtype(dt, np.floating) for dt in X_train_processed.dtypes)
all_test_numeric = all(np.issubdtype(dt, np.floating) for dt in X_test_processed.dtypes)

assert all_train_numeric, "CRITICAL ERROR: Non-numeric columns present in X_train_processed!"
assert all_test_numeric, "CRITICAL ERROR: Non-numeric columns present in X_test_processed!"

print("Numerical Data Type Check: PASSED (All 16 columns are float64)")


X_train_processed data types: {dtype('float64'): 16}
X_test_processed data types : {dtype('float64'): 16}

Numerical Data Type Check: PASSED (All 16 columns are float64)


## 6. Target Separation & Leakage Prevention

### WHAT are we checking?
We confirm that:
1. Target column `default` is **NOT** present in `X_train_processed` or `X_test_processed`.
2. No target-derived feature exists in the feature matrix.
3. `y_train` and `y_test` are strictly segregated label series.

### WHY are we checking it?
If the target variable is accidentally left inside $X$, the model achieves 100% artificial accuracy by simply reading the label column. This is the most dangerous form of data leakage.

### WHAT does the result mean?
The target variable `default` is strictly quarantined in `y_train` and `y_test`.

In [13]:
# Check 5: Target Isolation
target_in_train_X = ('default' in X_train_processed.columns)
target_in_test_X = ('default' in X_test_processed.columns)

print(f"Target 'default' in X_train_processed: {target_in_train_X}")
print(f"Target 'default' in X_test_processed : {target_in_test_X}")
print(f"y_train shape: {y_train.shape} | Classes: {sorted(y_train.unique().tolist())}")
print(f"y_test shape : {y_test.shape}  | Classes: {sorted(y_test.unique().tolist())}\n")

assert not target_in_train_X, "CRITICAL ERROR: 'default' target found inside X_train_processed!"
assert not target_in_test_X, "CRITICAL ERROR: 'default' target found inside X_test_processed!"
assert set(y_train.unique()).issubset({0, 1}), "CRITICAL ERROR: Unexpected values in y_train!"
assert set(y_test.unique()).issubset({0, 1}), "CRITICAL ERROR: Unexpected values in y_test!"

print("Target Separation Check: PASSED (Target strictly quarantined)")


Target 'default' in X_train_processed: False
Target 'default' in X_test_processed : False
y_train shape: (40,) | Classes: [0, 1]
y_test shape : (10,)  | Classes: [0, 1]

Target Separation Check: PASSED (Target strictly quarantined)


## 7. Identifier Exclusion Check (`applicant_id`)

### WHAT are we checking?
We confirm that database surrogate key `applicant_id` is completely absent from:
- `X`
- `X_train`
- `X_test`
- `X_train_processed`
- `X_test_processed`
- The generated feature list

### WHY are we checking it?
Database IDs are arbitrarily assigned sequence numbers. If fed to a machine learning model:
- Tree-based algorithms split on arbitrary ID numbers (e.g. `applicant_id <= 25`), memorizing individual sample identities instead of learning real financial patterns.
- New production loan applicants will have unseen IDs (`51, 52, ...`) where learned rules are meaningless.

### WHAT does the result mean?
`applicant_id` is completely excluded from the ML feature matrix.

In [14]:
# Check 6: Identifier Exclusion
id_in_features = ('applicant_id' in X_train_processed.columns)
id_in_raw_X = ('applicant_id' in X.columns)

print(f"applicant_id in raw X            : {id_in_raw_X}")
print(f"applicant_id in X_train_processed: {id_in_features}\n")

assert not id_in_raw_X, "CRITICAL ERROR: 'applicant_id' found in raw X!"
assert not id_in_features, "CRITICAL ERROR: 'applicant_id' found in processed features!"

print("Identifier Exclusion Check: PASSED (applicant_id completely excluded)")


applicant_id in raw X            : False
applicant_id in X_train_processed: False

Identifier Exclusion Check: PASSED (applicant_id completely excluded)


## 8. Identity Risk vs. Credit Default Risk Architecture Audit

### 🛡️ Core System Principle:
```
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                                APPLICATION WORKFLOW                                      │
├─────────────────────────────────────────────┬────────────────────────────────────────────┤
│ 1. IDENTITY & ELIGIBILITY RISK              │ 2. CREDIT DEFAULT RISK (ML MODEL)          │
│    (src/verification/)                      │    (src/credit/ & ML Pipeline)             │
├─────────────────────────────────────────────┼────────────────────────────────────────────┤
│ • pan_verified                              │ • credit_score                             │
│ • kyc_verified                              │ • debt_to_income_ratio                     │
│ • mobile_verified                           │ • monthly_income & existing_emi            │
│ • identity_match_status                     │ • late_payment_count & previous_defaults   │
│ • identity_match_score                      │ • banking balance & cashflows              │
├─────────────────────────────────────────────┼────────────────────────────────────────────┤
│ Purpose: ELIGIBILITY GATE                   │ Purpose: RISK PROBABILITY & PRICING        │
│ Decision: Reject fraud / Manual KYC review  │ Output: Probability of Default (PD)        │
│ Gating: Runs BEFORE credit underwriting     │ Evaluates: Willingness & Ability to Repay  │
└─────────────────────────────────────────────┴────────────────────────────────────────────┘
```

#### WHAT are we checking?
We verify that none of the identity verification flags have accidentally been included in the default-risk ML feature vector.

#### WHY are we checking it?
- A failed PAN/Aadhaar check means an applicant is unverified or potentially fraudulent—it is **NOT** a calibrated statistical predictor of loan repayment capacity.
- Combining fraud detection and credit default risk into a single statistical model destroys probability calibration and introduces severe confounding bias.

#### WHAT does the result mean?
Identity risk signals remain safely housed in the product/eligibility layer, and the ML feature vector contains exclusively credit default signals.

In [15]:
# Check 7: Verification / Identity Signal Audit
verification_fields = [
    'pan_verified', 'kyc_verified', 'mobile_verified', 
    'identity_match_status', 'identity_match_score',
    'aadhaar_verified', 'pan_status', 'name_match_score'
]

leaked_verification_fields = [f for f in verification_fields if f in X_train_processed.columns]

print(f"Verification signals tested: {len(verification_fields)}")
print(f"Verification signals found in ML feature vector: {leaked_verification_fields}\n")

assert len(leaked_verification_fields) == 0, f"CRITICAL ARCHITECTURAL LEAK: Found {leaked_verification_fields} in ML vector!"

print("Identity Risk Separation Audit: PASSED (ML vector contains ONLY credit risk features)")


Verification signals tested: 8
Verification signals found in ML feature vector: []

Identity Risk Separation Audit: PASSED (ML vector contains ONLY credit risk features)


## 9. Feature Redundancy & Multicollinearity Review

### WHAT are we checking?
We examine correlation relationships among related financial features:
- `monthly_income` vs. `existing_emi` vs. `debt_to_income_ratio`
- `loan_amount` vs. `loan_term_months` vs. `monthly_expenses`

### WHY are we checking it?
In financial credit risk modeling, some features are mathematically or behaviorally related:
1. **`debt_to_income_ratio` is derived from `existing_emi / monthly_income`**:
   - `monthly_income` measures absolute earning capacity (scale).
   - `existing_emi` measures total monthly liability load (obligation).
   - `debt_to_income_ratio` measures the relative financial strain (burden percentage).
   - Even though these three metrics are correlated, each provides a distinct financial signal: a borrower earning ₹1,00,000 with ₹30,000 EMI has the same DTI (0.30) as someone earning ₹30,000 with ₹9,000 EMI, but drastically different disposable income cushions.
2. **`loan_amount` and `loan_term_months`**:
   - Larger loans naturally require longer tenures to keep EMIs manageable.
3. **`monthly_income` and `monthly_expenses`**:
   - Higher earners naturally have higher living expenses (lifestyle scale).

### ⚠️ IMPORTANT DESIGN DECISION:
**We do NOT automatically delete correlated features at this stage.**
- Tree-based models (XGBoost, Random Forest) are largely robust to multicollinearity.
- Regularized linear models ($L_1 / L_2$ Logistic Regression) handle correlated features gracefully.
- Feature pruning decisions should be guided by model feature importance and SHAP analysis in Phase 3, not premature manual deletion.

### WHAT does the result mean?
The correlation matrix reveals expected, domain-logical relationships with no unhandled artificial duplicates.

In [16]:
# Check 8: Feature Redundancy Correlation Matrix
redundancy_subset = [
    'monthly_income', 'existing_emi', 'debt_to_income_ratio',
    'loan_amount', 'loan_term_months', 'monthly_expenses', 'credit_score'
]

raw_corr = X_train[redundancy_subset].corr()

print("=== Correlation Matrix (Selected Related Features) ===")
display(raw_corr.round(3))


=== Correlation Matrix (Selected Related Features) ===


,monthly_income,existing_emi,debt_to_income_ratio,loan_amount,loan_term_months,monthly_expenses,credit_score
monthly_income,1.000,0.807,-0.593,0.968,0.895,0.981,0.934
existing_emi,0.807,1.000,-0.026,0.889,0.822,0.868,0.629
debt_to_income_ratio,-0.593,-0.026,1.000,-0.421,-0.416,-0.470,-0.722
loan_amount,0.968,0.889,-0.421,1.000,0.895,0.987,0.857
loan_term_months,0.895,0.822,-0.416,0.895,1.000,0.881,0.822
monthly_expenses,0.981,0.868,-0.470,0.987,0.881,1.000,0.879
credit_score,0.934,0.629,-0.722,0.857,0.822,0.879,1.000


## 10. Final Model-Input Validation Gates & Assertions

### WHAT are we checking?
We execute an end-to-end automated assertion suite that validates all 10 model-ready conditions simultaneously.

### WHY are we checking it?
To serve as an automated, non-negotiable gatekeeper before Phase 3 model training.

### WHAT does the result mean?
All 10 validation gates pass with 100% compliance. The dataset is fully validated and certified model-ready.

In [17]:
# Check 9: Comprehensive Model-Ready Gate Suite
print("Executing Final Model-Ready Validation Gates...")

# Gate 1: Row count integrity
assert len(X_train_processed) == len(y_train) == 40, "Gate 1 Failed: Train row mismatch"
assert len(X_test_processed) == len(y_test) == 10, "Gate 1 Failed: Test row mismatch"

# Gate 2: Column count integrity
assert X_train_processed.shape[1] == X_test_processed.shape[1] == 16, "Gate 2 Failed: Feature count mismatch"

# Gate 3: Schema alignment
assert list(X_train_processed.columns) == list(X_test_processed.columns), "Gate 3 Failed: Feature name mismatch"

# Gate 4: Zero missing values
assert X_train_processed.isna().sum().sum() == 0, "Gate 4 Failed: NaNs in X_train"
assert X_test_processed.isna().sum().sum() == 0, "Gate 4 Failed: NaNs in X_test"

# Gate 5: Zero infinite values
assert np.isinf(X_train_processed.values).sum() == 0, "Gate 5 Failed: Infs in X_train"
assert np.isinf(X_test_processed.values).sum() == 0, "Gate 5 Failed: Infs in X_test"

# Gate 6: Strict floating-point dtype
assert all(np.issubdtype(t, np.floating) for t in X_train_processed.dtypes), "Gate 6 Failed: Non-float dtypes in train"
assert all(np.issubdtype(t, np.floating) for t in X_test_processed.dtypes), "Gate 6 Failed: Non-float dtypes in test"

# Gate 7: Target isolation
assert 'default' not in X_train_processed.columns, "Gate 7 Failed: Target leaked into train features"
assert 'default' not in X_test_processed.columns, "Gate 7 Failed: Target leaked into test features"

# Gate 8: Identifier isolation
assert 'applicant_id' not in X_train_processed.columns, "Gate 8 Failed: applicant_id found in features"

# Gate 9: Identity/KYC isolation
assert all(v not in X_train_processed.columns for v in verification_fields), "Gate 9 Failed: Verification flags in features"

# Gate 10: Target validity
assert set(y_train.unique()) == {0, 1}, "Gate 10 Failed: Invalid training target classes"
assert set(y_test.unique()) == {0, 1}, "Gate 10 Failed: Invalid test target classes"

print("=" * 60)
print("  ALL 10 MODEL-READY GATES PASSED SUCCESSFULLY!")
print("=" * 60)


Executing Final Model-Ready Validation Gates...
  ALL 10 MODEL-READY GATES PASSED SUCCESSFULLY!


## 11. Final Model Input Summary & Phase 2 Conclusion

### 📋 Model-Input Summary Specification:

| Property | Value | Notes |
|---|---|---|
| **Raw ML Features Count** | **14** | 13 numerical + 1 categorical |
| **Processed ML Features Count** | **16** | 13 scaled numerical + 3 one-hot encoded levels |
| **Training Records (`X_train_processed`)** | **40** | 80% stratified split |
| **Testing Records (`X_test_processed`)** | **10** | 20% stratified split |
| **Target Variable ($y$)** | `default` | Binary integer (0 = Non-default, 1 = Default) |
| **Train Target Balance** | 25 Non-default (62.5%) / 15 Default (37.5%) | Preserved via stratification |
| **Test Target Balance** | 6 Non-default (60.0%) / 4 Default (40.0%) | Preserved via stratification |
| **Excluded Identifier** | `applicant_id` | Surrogate key excluded to prevent memorization |
| **Excluded Product/KYC Signals** | `pan_verified`, `kyc_verified`, etc. | Gated in workflow layer (Identity Risk $\neq$ Credit Risk) |
| **Numerical Transformer** | `StandardScaler()` | Fitted **only** on `X_train` ($\mu \approx 0, \sigma = 1$) |
| **Categorical Transformer** | `OneHotEncoder(handle_unknown='ignore')` | Fitted **only** on `X_train` |
| **Missing / Infinite Values** | **0** | Clean, complete floating-point matrix |
| **Data Leakage Status** | **ZERO LEAKAGE** | Strict split-before-fit protocol enforced |

---

### Generated 16 ML Feature Names (Exact Order):
1. `age`
2. `monthly_income`
3. `loan_amount`
4. `loan_term_months`
5. `existing_emi`
6. `credit_score`
7. `credit_utilization_pct`
8. `late_payment_count`
9. `previous_defaults`
10. `employment_years`
11. `average_bank_balance`
12. `monthly_expenses`
13. `debt_to_income_ratio`
14. `employment_type_Business`
15. `employment_type_Salaried`
16. `employment_type_Self-employed`

---

### ⏭️ What to Understand Before Phase 3:
In **Phase 3 (Credit Risk ML Modeling & Probability of Default Estimation)**, we will:
1. Train baseline benchmark models (e.g., Logistic Regression with regularization).
2. Train non-linear tree-based models (e.g., Random Forest, XGBoost).
3. Generate calibrated continuous **Probability of Default (PD)** scores ($0.0 \le \text{PD} \le 1.0$).
4. Map PD to a standardized 0–100 Credit Risk Score.
5. Evaluate performance using ROC-AUC, PR-AUC, Precision, Recall, and Brier Score.

---

**PHASE 2 — STEP 3 COMPLETE.**  
**PHASE 2 COMPLETE.**  
*PHASE 3 HAS NOT BEEN STARTED.*